In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_raw = mnist.data.astype(float) / 255.0   # shape: (70000, 784)
y = mnist.target.astype(int)        # shape: (70000,)

print("X shape:", X_raw.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))

X shape: (70000, 784)
y shape: (70000,)
Unique labels: [0 1 2 3 4 5 6 7 8 9]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=10000, random_state=42, stratify=y
)

X_train_input = X_train  # shape (60000, 784)
X_test_input  = X_test

from sklearn.decomposition import PCA
pca = PCA(n_components=100)
X_train_input = pca.fit_transform(X_train)
X_test_input  = pca.transform(X_test)

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Convert numpy arrays to PyTorch tensors
X_tr = torch.FloatTensor(X_train_input)
y_tr = torch.LongTensor(y_train)
X_te = torch.FloatTensor(X_test_input)
y_te = torch.LongTensor(y_test)

# Define the network
class FNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)   # 10 output classes
        )
    
    def forward(self, x):
        return self.net(x)

model = FNN(input_dim=X_tr.shape[1])

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()  # combines softmax + log-loss

# Mini-batch training
dataset = TensorDataset(X_tr, y_tr)
loader  = DataLoader(dataset, batch_size=256, shuffle=True)

train_losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()          # clear old gradients
        preds = model(X_batch)         # forward pass
        loss  = loss_fn(preds, y_batch)
        loss.backward()                # backpropagation
        optimizer.step()               # update weights
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(loader))
    print(f"Epoch {epoch+1}/20 | Loss: {train_losses[-1]:.4f}")

Epoch 1/20 | Loss: 0.4684
Epoch 2/20 | Loss: 0.1317
Epoch 3/20 | Loss: 0.0841
Epoch 4/20 | Loss: 0.0597
Epoch 5/20 | Loss: 0.0441
Epoch 6/20 | Loss: 0.0324
Epoch 7/20 | Loss: 0.0240
Epoch 8/20 | Loss: 0.0184
Epoch 9/20 | Loss: 0.0143
Epoch 10/20 | Loss: 0.0104
Epoch 11/20 | Loss: 0.0077
Epoch 12/20 | Loss: 0.0058
Epoch 13/20 | Loss: 0.0040
Epoch 14/20 | Loss: 0.0037
Epoch 15/20 | Loss: 0.0030
Epoch 16/20 | Loss: 0.0022
Epoch 17/20 | Loss: 0.0013
Epoch 18/20 | Loss: 0.0009
Epoch 19/20 | Loss: 0.0007
Epoch 20/20 | Loss: 0.0006


In [6]:
model.eval()  # turns off dropout/batchnorm if you have them
with torch.no_grad():  # no need to compute gradients for evaluation
    train_preds = model(X_tr).argmax(dim=1)
    test_preds  = model(X_te).argmax(dim=1)

train_acc = (train_preds == y_tr).float().mean().item()
test_acc  = (test_preds  == y_te).float().mean().item()
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test  accuracy: {test_acc:.4f}")

Train accuracy: 1.0000
Test  accuracy: 0.9823


In [8]:
for d1, d2 in [(4, 9), (1, 6)]:
    mask_tr = np.isin(y_train, [d1, d2])
    mask_te = np.isin(y_test,  [d1, d2])
    # ... build tensors, train FNN with output_dim=2, evaluate
    X_tr_bin = torch.FloatTensor(X_train_input[mask_tr])
    y_tr_bin = torch.LongTensor((y_train[mask_tr] == d2).astype(int))  # 0 for d1, 1 for d2
    X_te_bin = torch.FloatTensor(X_test_input[mask_te])
    y_te_bin = torch.LongTensor((y_test[mask_te] == d2).astype(int))
    model_bin = FNN(input_dim=X_tr_bin.shape[1])
    optimizer_bin = torch.optim.Adam(model_bin.parameters(), lr=1e-3)
    dataset_bin = TensorDataset(X_tr_bin, y_tr_bin)
    loader_bin  = DataLoader(dataset_bin, batch_size=256, shuffle=True)
    for epoch in range(20):
        model_bin.train()
        for X_batch, y_batch in loader_bin:
            optimizer_bin.zero_grad()
            preds = model_bin(X_batch)
            loss  = loss_fn(preds, y_batch)
            loss.backward()
            optimizer_bin.step()
    model_bin.eval()
    with torch.no_grad():
        train_preds_bin = model_bin(X_tr_bin).argmax(dim=1)
        test_preds_bin  = model_bin(X_te_bin).argmax(dim=1)
    train_acc_bin = (train_preds_bin == y_tr_bin).float().mean().item()
    test_acc_bin  = (test_preds_bin  == y_te_bin).float().mean().item()
    print(f"Binary classification for digits {d1} vs {d2}:")
    print(f"  Train accuracy: {train_acc_bin:.4f}")
    print(f"  Test  accuracy: {test_acc_bin:.4f}")

Binary classification for digits 4 vs 9:
  Train accuracy: 1.0000
  Test  accuracy: 0.9909
Binary classification for digits 1 vs 6:
  Train accuracy: 1.0000
  Test  accuracy: 0.9995


In [11]:
# Reshape from (N, 784) to (N, 1, 28, 28)
# 1 = grayscale channel, 28x28 = image dimensions
X_tr_cnn = torch.FloatTensor(X_train).reshape(-1, 1, 28, 28)
X_te_cnn = torch.FloatTensor(X_test).reshape(-1, 1, 28, 28)
y_tr_cnn = torch.LongTensor(y_train)
y_te_cnn = torch.LongTensor(y_test)

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),   # 1 input channel (grayscale), 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                    # 28x28 -> 13x13
            nn.Conv2d(32, 64, kernel_size=3),  # 32 -> 64 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                    # 13x13 -> 5x5
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                       # (64, 5, 5) -> 1600
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x)

cnn_model = CNN()
print(cnn_model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1600, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [13]:
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

dataset_cnn = TensorDataset(X_tr_cnn, y_tr_cnn)
loader_cnn  = DataLoader(dataset_cnn, batch_size=256, shuffle=True)

cnn_losses = []
for epoch in range(10):  # CNNs converge faster — 10 epochs is often enough
    cnn_model.train()
    epoch_loss = 0
    for X_batch, y_batch in loader_cnn:
        optimizer.zero_grad()
        preds = cnn_model(X_batch)
        loss  = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    cnn_losses.append(epoch_loss / len(loader_cnn))
    print(f"Epoch {epoch+1}/10 | Loss: {cnn_losses[-1]:.4f}")

Epoch 1/10 | Loss: 0.3737
Epoch 2/10 | Loss: 0.0822
Epoch 3/10 | Loss: 0.0549
Epoch 4/10 | Loss: 0.0415
Epoch 5/10 | Loss: 0.0356
Epoch 6/10 | Loss: 0.0288
Epoch 7/10 | Loss: 0.0254
Epoch 8/10 | Loss: 0.0211
Epoch 9/10 | Loss: 0.0175
Epoch 10/10 | Loss: 0.0157


In [14]:
cnn_model.eval()
with torch.no_grad():
    cnn_train_preds = cnn_model(X_tr_cnn).argmax(dim=1)
    cnn_test_preds  = cnn_model(X_te_cnn).argmax(dim=1)

cnn_train_acc = (cnn_train_preds == y_tr_cnn).float().mean().item()
cnn_test_acc  = (cnn_test_preds  == y_te_cnn).float().mean().item()
print(f"CNN Train accuracy: {cnn_train_acc:.4f}")
print(f"CNN Test  accuracy: {cnn_test_acc:.4f}")

CNN Train accuracy: 0.9974
CNN Test  accuracy: 0.9913


In [15]:
# comparison table between FNN and CNN
print("Model Comparison:")
print(f"{'Model':<10} {'Train Acc':<10} {'Test Acc':<10}")
print(f"{'FNN':<10} {train_acc:.4f}     {test_acc:.4f}")
print(f"{'CNN':<10} {cnn_train_acc:.4f}     {cnn_test_acc:.4f}")

Model Comparison:
Model      Train Acc  Test Acc  
FNN        1.0000     0.9823
CNN        0.9974     0.9913
